In [2]:
from pathlib import Path
import requests
import subprocess

output_dir = Path("/kaggle/working/ruod_download")
output_dir.mkdir(parents=True, exist_ok=True)

api_url = (
    "https://api.github.com/repos/xiaoDetection/RUOD/"
    "releases/tags/untagged-4f7c7ab75187d68b6449"
)

response = requests.get(api_url, timeout=30)
response.raise_for_status()
release = response.json()

assets = {
    asset["name"]: asset
    for asset in release.get("assets", [])
}

for name in ["RUOD.tar.partaa", "RUOD.tar.partab"]:
    if name not in assets:
        raise RuntimeError(
            f"{name} not found. Available files: {list(assets)}"
        )

    asset = assets[name]
    destination = output_dir / name
    expected_size = asset["size"]

    if (
        destination.exists()
        and destination.stat().st_size == expected_size
    ):
        print("Already downloaded:", name)
        continue

    print("Downloading:", name, flush=True)

    subprocess.run(
        [
            "curl",
            "--fail",
            "--location",
            "--retry", "3",
            "--connect-timeout", "30",
            "--continue-at", "-",
            "--output", str(destination),
            asset["browser_download_url"]
        ],
        check=True
    )

    if destination.stat().st_size != expected_size:
        raise RuntimeError(f"Incomplete download: {name}")

    print("Complete:", name, flush=True)

print("\nBoth RUOD parts downloaded successfully.")

Downloading: RUOD.tar.partaa


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 98 2000M   98 1961M    0     0   122M      0  0:00:16  0:00:16 --:--:--  136M

Complete: RUOD.tar.partaa
Downloading: RUOD.tar.partab


100 2000M  100 2000M    0     0   122M      0  0:00:16  0:00:16 --:--:--  132M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 95 1541M   95 1470M    0     0   115M      0  0:00:13  0:00:12  0:00:01  115M

Complete: RUOD.tar.partab

Both RUOD parts downloaded successfully.


100 1541M  100 1541M    0     0   115M      0  0:00:13  0:00:13 --:--:--  117M


In [3]:
from pathlib import Path
from collections import Counter
import shutil
import tarfile

download_dir = Path("/kaggle/working/ruod_download")
parts = [
    download_dir / "RUOD.tar.partaa",
    download_dir / "RUOD.tar.partab"
]

archive_path = download_dir / "RUOD.tar"
output_dir = Path("/kaggle/working/ruod_original")

for part in parts:
    if not part.exists():
        raise FileNotFoundError(part)

# Join the parts in the required order.
expected_size = sum(part.stat().st_size for part in parts)

if not (
    archive_path.exists()
    and archive_path.stat().st_size == expected_size
):
    print("Joining archive parts...")

    temporary = download_dir / "RUOD.tar.partial"

    with temporary.open("wb") as destination:
        for part in parts:
            with part.open("rb") as source:
                shutil.copyfileobj(
                    source, destination,
                    length=8 * 1024 * 1024
                )

    temporary.replace(archive_path)

print("Extracting...")
output_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(archive_path, mode="r:*") as archive:
    archive.extractall(path=output_dir, filter="data")

files = sorted(
    path for path in output_dir.rglob("*") if path.is_file()
)
counts = Counter(path.suffix.lower() for path in files)

print("\nEXTRACTION COMPLETE")
print("Folder:", output_dir)

print("\nFILE COUNTS")
for extension, count in counts.most_common(10):
    print(extension or "(no extension)", ":", count)

print("\nANNOTATION / CONFIG EXAMPLES")
for extension in [".xml", ".json", ".txt", ".yaml", ".yml"]:
    examples = [
        str(path.relative_to(output_dir))
        for path in files
        if path.suffix.lower() == extension
    ][:3]

    if examples:
        print(extension, ":", examples)

Joining archive parts...
Extracting...

EXTRACTION COMPLETE
Folder: /kaggle/working/ruod_original

FILE COUNTS
.jpg : 14300
.json : 5

ANNOTATION / CONFIG EXAMPLES
.json : ['RUOD/Environmet_ANN/instances_blur.json', 'RUOD/Environmet_ANN/instances_color.json', 'RUOD/Environmet_ANN/instances_light.json']


In [4]:
from pathlib import Path
from urllib.parse import urlencode
import subprocess
import zipfile
import hashlib

# Build URL without depending on an earlier cell.
clean_url = (
    "https" + "://" + "dataverse.no"
    + "/api/access/datafile/:persistentId?"
    + urlencode({
        "persistentId": "doi:10.18710/H5G3K5/QK355D"
    })
)

folder = Path("/kaggle/working/norfisk_download")
folder.mkdir(parents=True, exist_ok=True)
destination = folder / "NorFisk_v1.0.zip"

def md5_file(path):
    digest = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

expected_md5 = "6ec799e74054dec2dadb6f9179e51ea5"

if (
    destination.is_file()
    and md5_file(destination) == expected_md5
):
    print("Verified ZIP already exists.")
else:
    print("Downloading NorFisk...", flush=True)

    subprocess.run([
        "curl", "--fail", "--location",
        "--retry", "2",
        "--connect-timeout", "30",
        "--continue-at", "-",
        "--output", str(destination),
        clean_url,
    ], check=True)

    assert md5_file(destination) == expected_md5, (
        "Checksum mismatch. ZIP অসম্পূর্ণ বা ভিন্ন file। "
        "এখন extraction চালাবে না।"
    )

assert zipfile.is_zipfile(destination), "Not a valid ZIP."
print("NorFisk ZIP verified:", destination)

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100 2518M  100 2518M    0     0  18.4M      0  0:02:16  0:02:16 --:--:-- 17.1M


NorFisk ZIP verified: /kaggle/working/norfisk_download/NorFisk_v1.0.zip


In [5]:
from pathlib import Path
from zipfile import ZipFile
from collections import Counter

zip_path = Path(
    "/kaggle/working/norfisk_download/NorFisk_v1.0.zip"
)
output_dir = Path("/kaggle/working/norfisk_original")

if not zip_path.exists():
    raise FileNotFoundError(
        "এই session-এ ZIP নেই। Saved output input হিসেবে "
        "যোগ করতে হবে—আবার download এখনই কোরো না।"
    )

output_dir.mkdir(parents=True, exist_ok=True)

with ZipFile(zip_path) as archive:
    # Check paths before extraction.
    for member in archive.infolist():
        destination = (output_dir / member.filename).resolve()
        if not destination.is_relative_to(output_dir.resolve()):
            raise ValueError(f"Unsafe archive path: {member.filename}")

    print("Extracting NorFisk...")
    archive.extractall(output_dir)

files = sorted(
    path for path in output_dir.rglob("*") if path.is_file()
)
counts = Counter(path.suffix.lower() for path in files)

print("\nFILE COUNTS")
for extension, count in counts.most_common(10):
    print(extension or "(no extension)", ":", count)

print("\nEXAMPLE PATHS")
for extension in [".jpg", ".jpeg", ".png", ".csv", ".txt", ".xml", ".json"]:
    examples = [
        str(path.relative_to(output_dir))
        for path in files if path.suffix.lower() == extension
    ][:3]
    if examples:
        print(extension, ":", examples)

# Inspect a few small text/CSV annotation candidates.
candidates = [
    path for path in files
    if path.suffix.lower() in {".csv", ".txt"}
    and "annot" in str(path).lower()
]

for path in candidates[:3]:
    print("\nANNOTATION SAMPLE:", path.relative_to(output_dir))
    with path.open(errors="replace") as file:
        for _ in range(3):
            line = file.readline()
            if not line:
                break
            print(line.rstrip()[:500])

Extracting NorFisk...

FILE COUNTS
.txt : 12514
.png : 12514

EXAMPLE PATHS
.png : ['saithe/img/saithe.000001.png', 'saithe/img/saithe.000002.png', 'saithe/img/saithe.000003.png']
.txt : ['saithe/annotations/saithe.000001.txt', 'saithe/annotations/saithe.000002.txt', 'saithe/annotations/saithe.000003.txt']

ANNOTATION SAMPLE: saithe/annotations/saithe.000001.txt
saithe.000001.png,saithe,65,41,170,137,235,177

ANNOTATION SAMPLE: saithe/annotations/saithe.000002.txt
saithe.000002.png,saithe,76,50,196,155,271,204

ANNOTATION SAMPLE: saithe/annotations/saithe.000003.txt
saithe.000003.png,saithe,73,56,216,163,288,218


In [6]:
from pathlib import Path
from collections import Counter
import json
import csv
import pandas as pd

report_dir = Path("/kaggle/working/three_domain_audit")
report_dir.mkdir(parents=True, exist_ok=True)

class_rows = []
file_rows = []

def inspect_coco(dataset, path):
    data = json.loads(path.read_text())

    if not (
        isinstance(data, dict)
        and all(key in data for key in [
            "images", "annotations", "categories"
        ])
    ):
        return

    categories = {
        category["id"]: category["name"]
        for category in data["categories"]
    }
    counts = Counter(
        annotation["category_id"]
        for annotation in data["annotations"]
    )

    file_rows.append({
        "dataset": dataset,
        "annotation_file": str(path),
        "image_records": len(data["images"]),
        "boxes": len(data["annotations"])
    })

    for category_id, name in categories.items():
        class_rows.append({
            "dataset": dataset,
            "annotation_file": path.name,
            "class": name,
            "boxes": counts[category_id]
        })

# Brackish: read only ground-truth JSONs.
brackish_files = [
    path for path in Path("/kaggle/input").rglob("*groundtruth.json")
    if "brackish" in str(path).lower()
]

if not brackish_files:
    raise FileNotFoundError("Brackish dataset Input-এ add করো।")

for path in sorted(brackish_files):
    inspect_coco("Brackish", path)

# RUOD: keep each JSON separate to avoid double-counting subsets.
ruod_dir = Path("/kaggle/working/ruod_original")

if not ruod_dir.exists():
    raise FileNotFoundError("Extracted RUOD folder পাওয়া যায়নি।")

for path in sorted(ruod_dir.rglob("*.json")):
    inspect_coco("RUOD", path)

# NorFisk: filename,class,xmin,ymin,xmax,ymax,width,height.
norfisk_dir = Path("/kaggle/working/norfisk_original")
norfisk_counts = Counter()
norfisk_image_names = set()
annotation_count = 0

for path in sorted(norfisk_dir.rglob("*.txt")):
    if path.parent.name.lower() != "annotations":
        continue

    annotation_count += 1

    with path.open(newline="") as file:
        for row in csv.reader(file):
            if not row or not any(value.strip() for value in row):
                continue

            if len(row) != 8:
                raise ValueError(f"Unexpected annotation: {path}: {row}")

            # Verify that coordinate fields are numeric.
            [float(value) for value in row[2:]]

            norfisk_image_names.add(row[0].strip())
            norfisk_counts[row[1].strip()] += 1

for name, count in sorted(norfisk_counts.items()):
    class_rows.append({
        "dataset": "NorFisk",
        "annotation_file": "individual TXT files",
        "class": name,
        "boxes": count
    })

file_rows.append({
    "dataset": "NorFisk",
    "annotation_file": f"{annotation_count} TXT files",
    "image_records": len(norfisk_image_names),
    "boxes": sum(norfisk_counts.values())
})

classes = pd.DataFrame(class_rows)
inventory = pd.DataFrame(file_rows)

classes.to_csv(report_dir / "class_box_counts.csv", index=False)
inventory.to_csv(report_dir / "annotation_inventory.csv", index=False)

print("ANNOTATION FILE SUMMARY")
display(inventory)

print("\nBRACKISH — totals across original splits")
display(
    classes[classes["dataset"] == "Brackish"]
    .groupby("class", as_index=False)["boxes"].sum()
)

print("\nRUOD — each annotation file shown separately")
display(classes[classes["dataset"] == "RUOD"])

print("\nNORFISK")
display(classes[classes["dataset"] == "NorFisk"])

print("\nSaved reports:", report_dir)

ANNOTATION FILE SUMMARY


,dataset,annotation_file,image_records,boxes
0,Brackish,/kaggle/input/datasets/aalborguniversity/brack...,1239,3466
1,Brackish,/kaggle/input/datasets/aalborguniversity/brack...,9967,28518
2,Brackish,/kaggle/input/datasets/aalborguniversity/brack...,1238,3581
3,RUOD,/kaggle/working/ruod_original/RUOD/Environmet_...,100,690
4,RUOD,/kaggle/working/ruod_original/RUOD/Environmet_...,100,551
5,RUOD,/kaggle/working/ruod_original/RUOD/Environmet_...,100,487
6,RUOD,/kaggle/working/ruod_original/RUOD/RUOD_ANN/in...,4200,22969
7,RUOD,/kaggle/working/ruod_original/RUOD/RUOD_ANN/in...,9800,51935
8,NorFisk,12514 TXT files,12514,12514



BRACKISH — totals across original splits


,class,boxes
0,crab,12348
1,fish,3352
2,jellyfish,637
3,shrimp,548
4,small_fish,10768
5,starfish,7912



RUOD — each annotation file shown separately


,dataset,annotation_file,class,boxes
18,RUOD,instances_blur.json,holothurian,178
19,RUOD,instances_blur.json,echinus,186
20,RUOD,instances_blur.json,scallop,95
21,RUOD,instances_blur.json,starfish,76
22,RUOD,instances_blur.json,fish,48
23,RUOD,instances_blur.json,corals,32
24,RUOD,instances_blur.json,diver,43
25,RUOD,instances_blur.json,cuttlefish,5
26,RUOD,instances_blur.json,turtle,23
27,RUOD,instances_blur.json,jellyfish,4



NORFISK


,dataset,annotation_file,class,boxes
68,NorFisk,individual TXT files,saithe,3027
69,NorFisk,individual TXT files,salmonid,9487



Saved reports: /kaggle/working/three_domain_audit


In [7]:

from pathlib import Path
from PIL import Image
import csv
import math
import pandas as pd

root = Path("/kaggle/working/norfisk_original")
output = Path("/kaggle/working/three_domain_audit")
output.mkdir(parents=True, exist_ok=True)

records = []
issues = []

for annotation in sorted(root.rglob("*.txt")):
    if annotation.parent.name.lower() != "annotations":
        continue

    with annotation.open(newline="") as file:
        for row in csv.reader(file):
            if not row or not any(value.strip() for value in row):
                continue

            if len(row) != 8:
                issues.append({
                    "file": str(annotation),
                    "issue": "unexpected_row_format"
                })
                continue

            name, label = row[0].strip(), row[1].strip()
            image_path = annotation.parent.parent / "img" / name

            if not image_path.exists():
                issues.append({
                    "file": str(annotation),
                    "issue": "missing_image"
                })
                continue

            try:
                x1, y1, x2, y2, aw, ah = map(float, row[2:])

                with Image.open(image_path) as image:
                    width, height = image.size

                problems = []

                if not all(math.isfinite(v) for v in [
                    x1, y1, x2, y2, aw, ah
                ]):
                    problems.append("non_finite")
                else:
                    if (width, height) != (aw, ah):
                        problems.append("dimension_mismatch")

                    if x2 <= x1 or y2 <= y1:
                        problems.append("non_positive_box")

                    if (
                        x1 < 0 or y1 < 0
                        or x2 > width or y2 > height
                    ):
                        problems.append("out_of_bounds")

                for problem in problems:
                    issues.append({
                        "file": str(annotation),
                        "issue": problem
                    })

                records.append({
                    "image_path": str(image_path),
                    "annotation_path": str(annotation),
                    "original_class": label,
                    "width": width,
                    "height": height,
                    "xmin": x1, "ymin": y1,
                    "xmax": x2, "ymax": y2,
                    "status": "ok" if not problems else "|".join(problems)
                })

            except Exception as error:
                issues.append({
                    "file": str(annotation),
                    "issue": f"read_error: {error}"
                })

manifest = pd.DataFrame(records)
issue_table = pd.DataFrame(issues, columns=["file", "issue"])

manifest.to_csv(output / "norfisk_manifest.csv", index=False)
issue_table.to_csv(output / "norfisk_issues.csv", index=False)

print("Annotation rows matched to readable images:", len(manifest))
print("Unique matched images:", manifest["image_path"].nunique())
print("Issue entries:", len(issue_table))

if len(issue_table):
    display(issue_table["issue"].value_counts())
else:
    print("Image matching, dimensions and box boundaries: PASSED")

display(
    manifest.groupby("original_class").agg(
        images=("image_path", "nunique"),
        min_width=("width", "min"),
        max_width=("width", "max"),
        min_height=("height", "min"),
        max_height=("height", "max")
    )
)

print("Reports saved:", output)

Annotation rows matched to readable images: 12514
Unique matched images: 12514
Issue entries: 0
Image matching, dimensions and box boundaries: PASSED


,images,min_width,max_width,min_height,max_height
original_class,,,,,
saithe,3027,125,3190,94,2156
salmonid,9487,101,1916,77,1295


Reports saved: /kaggle/working/three_domain_audit


In [8]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
import json
import pandas as pd

source = Path("/kaggle/working/ruod_original")
audit = Path("/kaggle/working/three_domain_audit")
audit.mkdir(parents=True, exist_ok=True)

lookup = defaultdict(list)

for path in source.rglob("*"):
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        lookup[path.name].append(path)

issues = []
checked = 0

for split in ["train", "test"]:
    matches = [
        path for path in source.rglob(f"instances_{split}.json")
        if path.parent.name == "RUOD_ANN"
    ]

    if len(matches) != 1:
        raise RuntimeError(f"Cannot locate {split} annotation uniquely")

    data = json.loads(matches[0].read_text())

    for record in data["images"]:
        name = Path(record["file_name"]).name
        candidates = lookup.get(name, [])

        if len(candidates) != 1:
            issues.append({
                "split": split,
                "filename": name,
                "issue": "missing_or_ambiguous_image"
            })
            continue

        with Image.open(candidates[0]) as image:
            actual_width, actual_height = image.size
            orientation = image.getexif().get(274, 1)

        checked += 1
        expected_width = record["width"]
        expected_height = record["height"]

        if (
            actual_width != expected_width
            or actual_height != expected_height
        ):
            issues.append({
                "split": split,
                "filename": name,
                "issue": "dimension_mismatch",
                "image_path": str(candidates[0]),
                "json_width": expected_width,
                "json_height": expected_height,
                "actual_width": actual_width,
                "actual_height": actual_height,
                "exif_orientation": orientation,
                "width_height_swapped": (
                    actual_width == expected_height
                    and actual_height == expected_width
                )
            })

report = pd.DataFrame(issues)
report.to_csv(audit / "ruod_dimension_issues.csv", index=False)

print("Images checked:", checked)
print("Issue entries:", len(report))

if len(report):
    display(report.head(20))
else:
    print("No dimension mismatches found.")

print("Saved:", audit / "ruod_dimension_issues.csv")

Images checked: 13704
Issue entries: 297


,split,filename,issue,image_path,json_width,json_height,actual_width,actual_height,exif_orientation,width_height_swapped
0,train,008431.jpg,dimension_mismatch,/kaggle/working/ruod_original/RUOD/RUOD_pic/tr...,1024.0,683.0,683.0,1024.0,6.0,True
1,test,001464.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,test,001102.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,test,012390.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,test,000005.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,test,004650.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,test,003473.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,test,010399.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,test,004443.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,test,001475.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved: /kaggle/working/three_domain_audit/ruod_dimension_issues.csv


In [9]:
from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageOps
import json
import pandas as pd

source = Path("/kaggle/working/ruod_original")
audit = Path("/kaggle/working/three_domain_audit")

lookup = defaultdict(list)

for path in source.rglob("*"):
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        lookup[path.name].append(path)

resolved_rows = []
unresolved = []
duplicate_examples = []

for split in ["train", "test"]:
    json_files = [
        path for path in source.rglob(f"instances_{split}.json")
        if path.parent.name == "RUOD_ANN"
    ]
    if len(json_files) != 1:
        raise RuntimeError(f"Cannot locate {split} JSON uniquely")

    data = json.loads(json_files[0].read_text())

    for record in data["images"]:
        name = Path(record["file_name"]).name
        candidates = lookup.get(name, [])

        if len(candidates) > 1 and len(duplicate_examples) < 3:
            duplicate_examples.append({
                "filename": name,
                "paths": [str(path) for path in candidates]
            })

        # Prefer the original split inside the main RUOD_pic folder.
        preferred = [
            path for path in candidates
            if "RUOD_pic" in path.parts
            and split in path.relative_to(source).parts[:-1]
        ]

        if len(preferred) == 1:
            chosen = preferred[0]
        elif len(candidates) == 1:
            chosen = candidates[0]
        else:
            unresolved.append({
                "split": split,
                "filename": name,
                "issue": "path_not_resolved",
                "candidate_paths": [str(path) for path in candidates]
            })
            continue

        with Image.open(chosen) as image:
            raw_size = image.size
            orientation = image.getexif().get(274, 1)
            oriented_size = ImageOps.exif_transpose(image).size

        expected = (record["width"], record["height"])

        if oriented_size != expected:
            unresolved.append({
                "split": split,
                "filename": name,
                "issue": "dimensions_still_mismatch",
                "raw_size": raw_size,
                "oriented_size": oriented_size,
                "expected_size": expected
            })
            continue

        resolved_rows.append({
            "original_split": split,
            "image_id": record["id"],
            "filename": name,
            "original_path": str(chosen),
            "width": expected[0],
            "height": expected[1],
            "exif_orientation": orientation,
            "needs_exif_normalization": orientation != 1
        })

resolved = pd.DataFrame(resolved_rows)
resolved.to_csv(audit / "ruod_resolved_images.csv", index=False)

pd.DataFrame(unresolved).to_csv(
    audit / "ruod_unresolved_images.csv", index=False
)

print("Resolved images:", len(resolved))
print("Unresolved images:", len(unresolved))
print(
    "Images needing EXIF normalization:",
    int(resolved["needs_exif_normalization"].sum())
)

print("\nDuplicate filename examples:")
for example in duplicate_examples:
    print(example["filename"])
    for path in example["paths"]:
        print(" ", path)

if unresolved:
    print("\nUnresolved examples:")
    display(pd.DataFrame(unresolved).head(10))

Resolved images: 14000
Unresolved images: 0
Images needing EXIF normalization: 11

Duplicate filename examples:
001464.jpg
  /kaggle/working/ruod_original/RUOD/Environment_pic/light/001464.jpg
  /kaggle/working/ruod_original/RUOD/RUOD_pic/test/001464.jpg
001102.jpg
  /kaggle/working/ruod_original/RUOD/Environment_pic/blur/001102.jpg
  /kaggle/working/ruod_original/RUOD/RUOD_pic/test/001102.jpg
012390.jpg
  /kaggle/working/ruod_original/RUOD/Environment_pic/blur/012390.jpg
  /kaggle/working/ruod_original/RUOD/RUOD_pic/test/012390.jpg


In [10]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import json
import math

BASE = Path("/kaggle/working/lodo_ready")
AUDIT = BASE / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

all_records = []
corrections = []
exclusions = []

def export_image(domain, key, source_path, width, height,
                 boxes, group, original_split="unknown"):
    """Export one image and its fish-only labels."""
    source_path = Path(source_path)

    if not source_path.exists():
        raise FileNotFoundError(source_path)

    folder = BASE / domain
    images = folder / "images"
    labels = folder / "labels"
    images.mkdir(parents=True, exist_ok=True)
    labels.mkdir(parents=True, exist_ok=True)

    destination = images / f"{key}{source_path.suffix.lower()}"
    label_path = labels / f"{key}.txt"

    lines = []

    for box in boxes:
        x, y, w, h = map(float, box)

        if (
            not all(math.isfinite(v) for v in [x, y, w, h])
            or w <= 0 or h <= 0
        ):
            raise ValueError(f"Invalid box: {domain}/{key}: {box}")

        x1, y1 = max(0.0, x), max(0.0, y)
        x2 = min(float(width), x + w)
        y2 = min(float(height), y + h)

        if x2 <= x1 or y2 <= y1:
            raise ValueError(f"Outside box: {domain}/{key}: {box}")

        clipped = [x1, y1, x2-x1, y2-y1]

        if not np.allclose(box, clipped, atol=1e-6, rtol=0):
            corrections.append({
                "domain": domain, "image": key,
                "original": list(box), "clipped": clipped
            })

        lines.append(
            f"0 {(x1+x2)/(2*width):.8f} "
            f"{(y1+y2)/(2*height):.8f} "
            f"{(x2-x1)/width:.8f} {(y2-y1)/height:.8f}"
        )

    # Links avoid duplicating several GB in this session.
    if destination.is_symlink():
        if destination.resolve() != source_path.resolve():
            raise RuntimeError(f"Wrong existing link: {destination}")
    elif destination.exists():
        raise RuntimeError(f"Unexpected existing file: {destination}")
    else:
        destination.symlink_to(source_path.resolve())

    label_path.write_text("\n".join(lines) + ("\n" if lines else ""))

    all_records.append({
        "domain": domain,
        "key": key,
        "image_path": str(destination),
        "label_path": str(label_path),
        "source_path": str(source_path),
        "width": width,
        "height": height,
        "fish_boxes": len(lines),
        "group": str(group),
        "original_split": original_split
    })

# Use the NorFisk manifest already verified in Cell 9.
nf = pd.read_csv(
    "/kaggle/working/three_domain_audit/norfisk_manifest.csv"
)

if not nf["status"].eq("ok").all():
    raise RuntimeError("NorFisk manifest contains unresolved issues.")

for row in nf.itertuples(index=False):
    path = Path(row.image_path)
    export_image(
        domain="norfisk",
        key=path.stem,
        source_path=path,
        width=row.width,
        height=row.height,
        boxes=[[
            row.xmin, row.ymin,
            row.xmax-row.xmin, row.ymax-row.ymin
        ]],
        # Individual image identity; not a recording ID.
        group=path.stem
    )

print("NorFisk exported:", len(nf), "images")

NorFisk exported: 12514 images


In [11]:
from pathlib import Path
import shutil
import pandas as pd

usage = shutil.disk_usage("/kaggle/working")
print(f"Total storage: {usage.total / 2**30:.2f} GB")
print(f"Used storage:  {usage.used / 2**30:.2f} GB")
print(f"Free storage:  {usage.free / 2**30:.2f} GB")

archive_dir = Path("/kaggle/working/ruod_download")
rows = []

if archive_dir.exists():
    for p in sorted(archive_dir.iterdir()):
        if p.is_file():
            rows.append({
                "file": str(p),
                "size_GB": round(p.stat().st_size / 2**30, 3)
            })

display(pd.DataFrame(rows))

frame_dir = Path("/kaggle/working/lodo_ready/brackish_frames")
print(
    "Brackish PNG files currently present:",
    sum(1 for _ in frame_dir.glob("*.png"))
)

Total storage: 19.52 GB
Used storage:  15.48 GB
Free storage:  4.02 GB


,file,size_GB
0,/kaggle/working/ruod_download/RUOD.tar,3.458
1,/kaggle/working/ruod_download/RUOD.tar.partaa,1.953
2,/kaggle/working/ruod_download/RUOD.tar.partab,1.505


Brackish PNG files currently present: 0


In [12]:
from pathlib import Path
import shutil

# Confirm that extracted RUOD data is present.
ruod_root = Path("/kaggle/working/ruod_original/RUOD")

jpg_count = sum(1 for _ in ruod_root.rglob("*.jpg"))
main_annotations = list(ruod_root.rglob("instances_train.json"))
test_annotations = list(ruod_root.rglob("instances_test.json"))

assert jpg_count >= 14000, f"Unexpected RUOD image count: {jpg_count}"
assert main_annotations and test_annotations, "RUOD annotations missing."

# Remove only redundant downloaded archive files.
archive_dir = Path("/kaggle/working/ruod_download")

for name in ["RUOD.tar", "RUOD.tar.partaa", "RUOD.tar.partab"]:
    p = archive_dir / name
    if p.is_file():
        size = p.stat().st_size / 2**30
        p.unlink()
        print(f"Removed: {name} ({size:.2f} GB)")

# Remove the PNG whose write failed, so extraction recreates it.
failed_frame = Path(
    "/kaggle/working/lodo_ready/brackish_frames/"
    "2019-03-21_03-21-54to2019-03-21_03-22-09_1-0120.png"
)
failed_frame.unlink(missing_ok=True)

print(
    f"\nFree storage: "
    f"{shutil.disk_usage('/kaggle/working').free / 2**30:.2f} GB"
)

Removed: RUOD.tar (3.46 GB)
Removed: RUOD.tar.partaa (1.95 GB)
Removed: RUOD.tar.partab (1.51 GB)

Free storage: 10.94 GB


In [11]:
from pathlib import Path
import pandas as pd

BASE = Path("/kaggle/working/lodo_ready")
AUDIT = BASE / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

manifest_path = AUDIT / "all_images.csv"

print("Manifest exists:", manifest_path.is_file())
print("Export function available:", callable(globals().get("export_image")))
print("In-memory records:", len(globals().get("all_records", [])))

for domain in ["brackish", "ruod", "norfisk"]:
    image_dir = BASE / domain / "images"
    label_dir = BASE / domain / "labels"

    print(
        domain,
        "| readable images:", sum(p.is_file() for p in image_dir.glob("*")),
        "| labels:", sum(p.is_file() for p in label_dir.glob("*.txt")),
    )

if manifest_path.is_file():
    manifest = pd.read_csv(manifest_path)
    print("\nManifest columns:", manifest.columns.tolist())
    print("Duplicate keys:", manifest.duplicated(["domain", "key"]).sum())
    display(manifest.groupby("domain").size().rename("images"))

Manifest exists: False
Export function available: True
In-memory records: 12514
brackish | readable images: 0 | labels: 0
ruod | readable images: 0 | labels: 0
norfisk | readable images: 12514 | labels: 12514


In [ ]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
import cv2
import json
import pandas as pd
import shutil

BASE = Path("/kaggle/working/lodo_ready")
AUDIT = BASE / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

# Keep the already exported NorFisk records.
# This also makes rerunning this cell safe for the in-memory manifest.
all_records = [
    r for r in all_records if r["domain"] == "norfisk"
]
assert len(all_records) == 12514, "Run the NorFisk export cell first."

# ---------- RUOD ----------
ruod_source = Path("/kaggle/working/ruod_original")
resolved_path = Path(
    "/kaggle/working/three_domain_audit/ruod_resolved_images.csv"
)
assert resolved_path.is_file(), "Run the RUOD path-resolution cell first."

resolved = pd.read_csv(resolved_path)
lookup = {
    (r.original_split, r.filename): r
    for r in resolved.itertuples(index=False)
}

for split in ["train", "test"]:
    matches = [
        p for p in ruod_source.rglob(f"instances_{split}.json")
        if p.parent.name == "RUOD_ANN"
    ]
    assert len(matches) == 1, f"Cannot locate RUOD {split} annotations."

    data = json.loads(matches[0].read_text())
    fish_ids = {
        c["id"] for c in data["categories"]
        if c["name"].strip().lower() == "fish"
    }

    boxes = defaultdict(list)
    for ann in data["annotations"]:
        if ann["category_id"] in fish_ids:
            boxes[ann["image_id"]].append(ann["bbox"])

    for record in data["images"]:
        filename = Path(record["file_name"]).name

        # Previously identified orientation/box-alignment issue.
        if filename == "008431.jpg":
            exclusions.append({
                "domain": "ruod",
                "image": filename,
                "reason": "Unresolved orientation/annotation alignment",
            })
            continue

        row = lookup[(split, filename)]
        orientation = int(row.exif_orientation)
        assert orientation in [0, 1], (
            f"Review orientation: {filename}, EXIF={orientation}"
        )

        source = Path(row.original_path)
        with Image.open(source) as image:
            actual_size = image.size

        assert actual_size == (record["width"], record["height"]), (
            f"Dimension mismatch: {filename}"
        )

        key = f"{split}_{Path(filename).stem}"
        export_image(
            domain="ruod",
            key=key,
            source_path=source,
            width=record["width"],
            height=record["height"],
            boxes=boxes[record["id"]],
            group=key,
            original_split=split,
        )

    print("RUOD exported:", split, flush=True)

# ---------- BRACKISH ----------
matches = [
    p for p in Path("/kaggle/input").rglob("train_groundtruth.json")
    if "brackish" in str(p).lower()
]
assert len(matches) == 1, "Add the Brackish dataset to notebook inputs."

annotation_dir = matches[0].parent
brackish_root = annotation_dir.parent.parent

records_by_name = {}
fish_boxes = defaultdict(list)

for split in ["train", "valid", "test"]:
    data = json.loads(
        (annotation_dir / f"{split}_groundtruth.json").read_text()
    )
    fish_ids = {
        c["id"] for c in data["categories"]
        if c["name"].strip().lower() in {"fish", "small_fish"}
    }
    id_to_name = {}

    for record in data["images"]:
        filename = Path(record["file_name"]).name
        assert filename not in records_by_name, (
            f"Repeated Brackish image record: {filename}"
        )
        records_by_name[filename] = (record, split)
        id_to_name[record["id"]] = filename

    for ann in data["annotations"]:
        if ann["category_id"] in fish_ids:
            fish_boxes[id_to_name[ann["image_id"]]].append(ann["bbox"])

videos = defaultdict(list)
for p in brackish_root.rglob("*.avi"):
    videos[p.stem].append(p)

frames_by_video = defaultdict(dict)
for filename in records_by_name:
    video_id, number = Path(filename).stem.rsplit("-", 1)
    frames_by_video[video_id][int(number)] = filename

frame_dir = BASE / "brackish_frames"
frame_dir.mkdir(parents=True, exist_ok=True)

for index, (video_id, wanted) in enumerate(
    sorted(frames_by_video.items()), start=1
):
    assert len(videos[video_id]) == 1, f"Video missing/ambiguous: {video_id}"

    pending = {}
    for number, filename in wanted.items():
        destination = frame_dir / filename

        if destination.exists():
            existing = cv2.imread(str(destination))
            if existing is not None and existing.shape[:2] == (540, 960):
                continue
            destination.unlink()

        pending[number] = filename

    if pending:
        capture = cv2.VideoCapture(str(videos[video_id][0]))
        assert capture.isOpened(), f"Cannot open video: {video_id}"
        number = 0

        try:
            while pending:
                ok, frame = capture.read()
                if not ok:
                    break
                number += 1

                if number not in pending:
                    continue

                resized = cv2.resize(
                    frame, (960, 540), interpolation=cv2.INTER_CUBIC
                )
                destination = frame_dir / pending[number]

                if not cv2.imwrite(str(destination), resized):
                    destination.unlink(missing_ok=True)
                    raise RuntimeError(
                        "Frame write failed. Free storage: "
                        f"{shutil.disk_usage(BASE).free / 2**30:.2f} GB"
                    )

                del pending[number]
        finally:
            capture.release()

        assert not pending, f"Missing frames in {video_id}: {len(pending)}"

    for filename in wanted.values():
        record, split = records_by_name[filename]
        assert (record["width"], record["height"]) == (960, 540)

        export_image(
            domain="brackish",
            key=Path(filename).stem,
            source_path=frame_dir / filename,
            width=960,
            height=540,
            boxes=fish_boxes[filename],
            group=video_id[:10],  # Recording date
            original_split=split,
        )

    if index % 10 == 0 or index == len(frames_by_video):
        print(
            f"Brackish videos: {index}/{len(frames_by_video)}",
            flush=True,
        )

# ---------- SAVE MANIFEST AND LOGS ----------
manifest = pd.DataFrame(all_records)
assert not manifest.duplicated(["domain", "key"]).any()

assert manifest.groupby("domain").size().to_dict() == {
    "brackish": 12444,
    "norfisk": 12514,
    "ruod": 13999,
}

manifest.to_csv(AUDIT / "all_images.csv", index=False)

def save_log(records, filename):
    unique = {}
    for record in records:
        signature = json.dumps(record, sort_keys=True, default=str)
        unique[signature] = record
    pd.DataFrame(unique.values()).to_csv(AUDIT / filename, index=False)

save_log(corrections, "box_corrections.csv")
save_log(exclusions, "excluded_images.csv")

display(
    manifest.groupby("domain").agg(
        images=("key", "size"),
        fish_boxes=("fish_boxes", "sum"),
    )
)
print("EXPORT COMPLETE — manifest saved.")
print(
    "Free storage:",
    round(shutil.disk_usage(BASE).free / 2**30, 2),
    "GB",
)

RUOD exported: train
RUOD exported: test


In [13]:
from pathlib import Path
import shutil

WORK = Path("/kaggle/working")

# Only remove download archives after extracted images exist.
checks = [
    (
        WORK / "ruod_download",
        WORK / "ruod_original",
        {".jpg", ".jpeg"},
        14300,
    ),
    (
        WORK / "norfisk_download",
        WORK / "norfisk_original",
        {".png"},
        12514,
    ),
]

for download_dir, extracted_dir, extensions, expected in checks:
    count = sum(
        1 for p in extracted_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in extensions
    )

    print(f"{extracted_dir.name}: {count} images")

    if count < expected:
        print("Extraction incomplete — archives kept.")
        continue

    if download_dir.exists():
        for p in download_dir.iterdir():
            is_archive = (
                p.name.endswith((".zip", ".tar", ".tar.gz"))
                or ".tar.part" in p.name
            )
            if p.is_file() and is_archive:
                size = p.stat().st_size / 1024**3
                p.unlink()
                print(f"Deleted download archive: {p.name} | {size:.2f} GB")

print(
    "\nFree storage:",
    round(shutil.disk_usage(WORK).free / 1024**3, 2),
    "GB",
)

ruod_original: 14300 images
Deleted download archive: RUOD.tar.partaa | 1.95 GB
Deleted download archive: RUOD.tar | 3.46 GB
Deleted download archive: RUOD.tar.partab | 1.51 GB
norfisk_original: 12514 images
Deleted download archive: NorFisk_v1.0.zip | 2.46 GB

Free storage: 9.38 GB


In [15]:
import hashlib
from sklearn.model_selection import GroupShuffleSplit
import yaml

manifest = pd.read_csv(AUDIT / "all_images.csv")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hashes = []
for index, path in enumerate(manifest["source_path"], start=1):
    hashes.append(sha256_file(path))
    if index % 5000 == 0:
        print("Files hashed:", index)

manifest["sha256"] = hashes

cross_domain = (
    manifest.groupby("sha256")["domain"].nunique()
)
overlap_hashes = cross_domain[cross_domain > 1].index

manifest.to_csv(AUDIT / "all_images_hashed.csv", index=False)

if len(overlap_hashes):
    manifest[manifest["sha256"].isin(overlap_hashes)].to_csv(
        AUDIT / "cross_domain_duplicates.csv", index=False
    )
    raise RuntimeError(
        "Cross-domain duplicates found. Review before training."
    )

manifest["source_partition"] = ""

for domain in ["brackish", "ruod", "norfisk"]:
    domain_rows = manifest[manifest["domain"] == domain]

    if domain == "brackish":
        # Union date groups connected by any exact duplicate.
        parent = {group: group for group in domain_rows["group"].unique()}

        def find(group):
            while parent[group] != group:
                parent[group] = parent[parent[group]]
                group = parent[group]
            return group

        for _, duplicates in domain_rows.groupby("sha256"):
            groups = duplicates["group"].unique()
            for group in groups[1:]:
                parent[find(group)] = find(groups[0])

        split_groups = domain_rows["group"].map(find).to_numpy()
    else:
        split_groups = domain_rows["sha256"].to_numpy()

    splitter = GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=42
    )
    train_index, val_index = next(
        splitter.split(domain_rows, groups=split_groups)
    )

    manifest.loc[
        domain_rows.index[train_index], "source_partition"
    ] = "train"
    manifest.loc[
        domain_rows.index[val_index], "source_partition"
    ] = "val"

    train_hashes = set(domain_rows.iloc[train_index]["sha256"])
    val_hashes = set(domain_rows.iloc[val_index]["sha256"])
    assert train_hashes.isdisjoint(val_hashes)

manifest.to_csv(AUDIT / "fixed_partitions.csv", index=False)

config_dir = BASE / "configs"
config_dir.mkdir(parents=True, exist_ok=True)

runs = [
    ("brackish_ruod_to_norfisk", ["brackish", "ruod"], "norfisk"),
    ("ruod_norfisk_to_brackish", ["ruod", "norfisk"], "brackish"),
    ("norfisk_brackish_to_ruod", ["norfisk", "brackish"], "ruod")
]

for name, sources, target in runs:
    train = manifest[
        manifest["domain"].isin(sources)
        & manifest["source_partition"].eq("train")
    ]
    val = manifest[
        manifest["domain"].isin(sources)
        & manifest["source_partition"].eq("val")
    ]
    test = manifest[manifest["domain"].eq(target)]

    lists = {}
    for split, table in [("train", train), ("val", val), ("test", test)]:
        path = config_dir / f"{name}_{split}.txt"
        path.write_text("\n".join(table["image_path"]) + "\n")
        lists[split] = str(path)

    config = {
        "path": str(BASE),
        **lists,
        "names": {0: "fish"}
    }

    (config_dir / f"{name}.yaml").write_text(
        yaml.safe_dump(config, sort_keys=False)
    )

    print(
        f"{name}: train={len(train)}, "
        f"source_val={len(val)}, unseen_test={len(test)}"
    )

display(
    manifest.groupby(["domain", "source_partition"]).agg(
        images=("key", "size"),
        fish_boxes=("fish_boxes", "sum")
    )
)

print("\nConfigs saved:", config_dir)
print("PREPARATION COMPLETE — no model trained yet.")

Files hashed: 5000
Files hashed: 10000
Files hashed: 15000
Files hashed: 20000
Files hashed: 25000
Files hashed: 30000
Files hashed: 35000
brackish_ruod_to_norfisk: train=20474, source_val=5969, unseen_test=12514
ruod_norfisk_to_brackish: train=21230, source_val=5283, unseen_test=12444
norfisk_brackish_to_ruod: train=19266, source_val=5692, unseen_test=13999


images  fish_boxes
domain   source_partition                    
brackish train               9255       10255
         val                 3189        3865
norfisk  train              10011       10011
         val                 2503        2503
ruod     train              11219       10655
         val                 2780        2436


Configs saved: /kaggle/working/lodo_ready/configs
PREPARATION COMPLETE — no model trained yet.


## Brackish দিয়ে training

In [17]:
%pip install -q ultralytics==8.4.207

Note: you may need to restart the kernel to use updated packages.


In [18]:
from pathlib import Path
import pandas as pd
import yaml
import torch
import shutil
import hashlib
import json
import ultralytics
from ultralytics import YOLO
from IPython.display import display, FileLink

BASE = Path("/kaggle/working/lodo_ready")
MANIFEST = BASE / "audit/fixed_partitions.csv"
RUN_NAME = "single_brackish_seed42"
PROJECT = Path("/kaggle/working/single_source_runs")
RUN_DIR = PROJECT / RUN_NAME
CONFIG_DIR = BASE / "single_source_configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Enable GPU first."
assert ultralytics.__version__ == "8.4.207", (
    "আগের version ব্যবহার করো: %pip install -q ultralytics==8.4.207"
)
assert MANIFEST.is_file(), f"Missing manifest: {MANIFEST}"
assert not RUN_DIR.exists(), f"Run already exists: {RUN_DIR}"

manifest = pd.read_csv(MANIFEST)
assert not manifest.duplicated(["domain", "key"]).any()

data = manifest[manifest["domain"] == "brackish"].copy()
assert len(data) == 12444
assert set(data["source_partition"]) == {"train", "val"}

# Preserve saved assignments; do not generate a new random split.
assert (
    data.groupby("group")["source_partition"].nunique() <= 1
).all(), "Recording group crosses train/validation."

assert (
    data.groupby("sha256")["source_partition"].nunique() <= 1
).all(), "Identical image crosses train/validation."

lists = {}

for partition, expected_count in [("train", 9255), ("val", 3189)]:
    subset = data[data["source_partition"] == partition]
    assert len(subset) == expected_count, (
        f"{partition}: expected {expected_count}, found {len(subset)}. "
        "আগের fixed_partitions.csv-এর সঙ্গে split মিলিয়ে নিতে হবে।"
    )

    paths = []
    for row in subset.itertuples(index=False):
        image = Path(row.image_path)
        label = Path(row.label_path)
        assert image.is_file(), f"Missing image: {image}"
        assert label.is_file(), f"Missing label: {label}"
        paths.append(str(image))

    listing = CONFIG_DIR / f"{RUN_NAME}_{partition}.txt"
    listing.write_text("\n".join(paths) + "\n")
    lists[partition] = listing

CONFIG = CONFIG_DIR / f"{RUN_NAME}.yaml"
CONFIG.write_text(yaml.safe_dump({
    "path": str(BASE),
    "train": str(lists["train"]),
    "val": str(lists["val"]),
    "names": {0: "fish"},
}))

print("GPU:", torch.cuda.get_device_name(0))
print("Source: Brackish only")
print("Train images: 9255")
print("Source validation images: 3189")
print("NorFisk / RUOD excluded from training and selection.", flush=True)

# Fresh pretrained initialization, NOT a previous trained best.pt.
model = YOLO("yolo11n.pt")

model.train(
    data=str(CONFIG),
    project=str(PROJECT),
    name=RUN_NAME,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    seed=42,
    deterministic=True,
    patience=30,
    save=True,
    save_period=5,
    plots=True,
    exist_ok=False,
)

# Save reproducibility metadata and portable small downloads.
actual_run = Path(model.trainer.save_dir)
data.to_csv(actual_run / "source_partitions.csv", index=False)
shutil.copy2(CONFIG, actual_run / "dataset_config.yaml")

for partition, listing in lists.items():
    shutil.copy2(listing, actual_run / f"{partition}_images.txt")

best = actual_run / "weights/best.pt"
assert best.is_file()

(actual_run / "experiment_identity.json").write_text(json.dumps({
    "source": "brackish",
    "seed": 42,
    "best_sha256": hashlib.sha256(best.read_bytes()).hexdigest(),
    "external_test_performed": False,
    "validation_role": "Source validation used for model selection",
}, indent=2))

prefix = "/kaggle/working/single_brackish"

for relative, suffix in {
    "weights/best.pt": "_best.pt",
    "weights/last.pt": "_last.pt",
    "results.csv": "_training_results.csv",
    "args.yaml": "_training_settings.yaml",
    "results.png": "_training_curves.png",
    "source_partitions.csv": "_source_partitions.csv",
}.items():
    source = actual_run / relative
    if source.is_file():
        destination = Path(prefix + suffix)
        shutil.copy2(source, destination)
        display(FileLink(str(destination)))

archive = shutil.make_archive(
    prefix + "_training_report", "zip", root_dir=actual_run
)
display(FileLink(archive))

print("\nBRACKISH-ONLY TRAINING COMPLETE")
print("Best model:", best)
print("Next: external evaluation on NorFisk and RUOD.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU: Tesla T4
Source: Brackish only
Train images: 9255
Source validation images: 3189
NorFisk / RUOD excluded from training and selection.
New https://pypi.org/project/ultralytics/8.4.214 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/lodo_ready/single_source_configs/si

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      2.57G      1.713      1.375      1.293          5        640: 100% ━━━━━━━━━━━━ 579/579 6.0it/s 1:360.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 5.2it/s 19.1s.2s
                   all       3189       3865      0.473      0.291      0.266      0.103

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      2.57G      1.709      1.319       1.29          7        640: 100% ━━━━━━━━━━━━ 579/579 6.1it/s 1:360.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 5.2it/s 19.2s.2s
                   all       3189       3865      0.501       0.41      0.388       0.16

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      2.57G      1.632      1.183      1.252          8        640: 100% ━━━━━━━━━━━━ 579/579 6.0it/s 1:360.3sss
                 Class    

/kaggle/working/single_brackish_best.pt

/kaggle/working/single_brackish_last.pt

/kaggle/working/single_brackish_training_results.csv

/kaggle/working/single_brackish_training_settings.yaml

/kaggle/working/single_brackish_training_curves.png

/kaggle/working/single_brackish_source_partitions.csv

/kaggle/working/single_brackish_training_report.zip


BRACKISH-ONLY TRAINING COMPLETE
Best model: /kaggle/working/single_source_runs/single_brackish_seed42/weights/best.pt
Next: external evaluation on NorFisk and RUOD.


## Test

In [19]:
from pathlib import Path
import json
import hashlib
import yaml
import pandas as pd
from ultralytics import YOLO
from IPython.display import display, FileLink

BASE = Path("/kaggle/working/lodo_ready")
WEIGHTS = Path(
    "/kaggle/working/single_source_runs/"
    "single_brackish_seed42/weights/best.pt"
)

assert WEIGHTS.is_file(), f"Checkpoint missing: {WEIGHTS}"

OUTPUT = Path("/kaggle/working/single_brackish_external_tests")
OUTPUT.mkdir(parents=True, exist_ok=True)

# These configs supply only the unseen target's test list.
TARGETS = {
    "norfisk": (
        BASE / "configs/brackish_ruod_to_norfisk.yaml",
        12514,
    ),
    "ruod": (
        BASE / "configs/norfisk_brackish_to_ruod.yaml",
        13999,
    ),
}

model_sha256 = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
rows = []

for target, (config_path, expected_images) in TARGETS.items():
    assert config_path.is_file(), f"Config missing: {config_path}"

    config = yaml.safe_load(config_path.read_text())
    assert "test" in config, f"No test entry: {config_path}"

    # Build a target-only evaluation config.
    test_config = {
        "path": config.get("path", str(BASE)),
        "train": config["test"],
        "val": config["test"],
        "test": config["test"],
        "names": {0: "fish"},
    }

    test_yaml = OUTPUT / f"{target}_test.yaml"
    test_yaml.write_text(
        yaml.safe_dump(test_config, sort_keys=False)
    )

    print(f"\nTesting Brackish-only model → {target.upper()}", flush=True)

    metrics = YOLO(str(WEIGHTS)).val(
        data=str(test_yaml),
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        workers=2,
        cache=False,
        conf=0.001,
        iou=0.7,
        max_det=300,
        augment=False,
        plots=True,
        save_json=True,
        save_txt=True,
        save_conf=True,
        project=str(OUTPUT),
        name=f"evaluation_{target}",
        exist_ok=False,
    )

    # Use native Ultralytics metrics consistently with earlier runs.
    row = {
        "Training domain": "Brackish",
        "Unseen test domain": target,
        "Precision (%)": float(metrics.box.mp) * 100,
        "Recall (%)": float(metrics.box.mr) * 100,
        "mAP50 (%)": float(metrics.box.map50) * 100,
        "mAP50-95 (%)": float(metrics.box.map) * 100,
    }
    rows.append(row)

    csv_path = OUTPUT / f"single_brackish_to_{target}_metrics.csv"
    pd.DataFrame([row]).to_csv(csv_path, index=False)

    details = {
        "model": str(WEIGHTS),
        "model_sha256": model_sha256,
        "training_domain": "brackish",
        "target_domain": target,
        "expected_test_images": expected_images,
        "checkpoint_selection": "Brackish source validation only",
        "evaluation_directory": str(metrics.save_dir),
        "metrics": row,
        "evaluation_settings": {
            "imgsz": 640,
            "conf": 0.001,
            "nms_iou": 0.7,
            "max_det": 300,
            "augment": False,
        },
    }

    json_path = OUTPUT / f"single_brackish_to_{target}_details.json"
    json_path.write_text(json.dumps(details, indent=2))

    # Update combined results after EACH completed target.
    pd.DataFrame(rows).to_csv(
        OUTPUT / "single_brackish_external_metrics.csv",
        index=False,
    )

    display(pd.DataFrame([row]).round(3))
    display(FileLink(str(csv_path)))
    display(FileLink(str(json_path)))

print("\nBOTH EXTERNAL TESTS COMPLETE")
display(pd.DataFrame(rows).round(3))
display(FileLink(str(OUTPUT / "single_brackish_external_metrics.csv")))


Testing Brackish-only model → NORFISK
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2323.6±1079.1 MB/s, size: 124.9 KB)
val: Scanning /kaggle/working/lodo_ready/norfisk/labels... 12514 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12514/12514 516.6it/s 24.2s0.0s
val: New cache created: /kaggle/working/lodo_ready/norfisk/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 783/783 5.4it/s 2:240.2ss
                   all      12514      12514     0.0691     0.0625     0.0148    0.00356
Speed: 0.6ms preprocess, 3.2ms inference, 0.0ms loss, 1.2ms postprocess per image
Saving /kaggle/working/single_brackish_external_tests/evaluation_norfisk/predictions.json...

Evaluating faster-coco-eval mAP...
requirements: Ultralytics requirement ['faster-coco-e

,Training domain,Unseen test domain,Precision (%),Recall (%),mAP50 (%),mAP50-95 (%)
0,Brackish,norfisk,6.906,6.249,1.482,0.356


/kaggle/working/single_brackish_external_tests/single_brackish_to_norfisk_metrics.csv

/kaggle/working/single_brackish_external_tests/single_brackish_to_norfisk_details.json


Testing Brackish-only model → RUOD
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1548.5±1263.9 MB/s, size: 86.2 KB)
val: Scanning /kaggle/working/lodo_ready/ruod/labels... 13999 images, 10834 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13999/13999 1.6Kit/s 9.0s0.0s
val: /kaggle/working/lodo_ready/ruod/images/train_003875.jpg: corrupt JPEG restored and saved
val: /kaggle/working/lodo_ready/ruod/images/train_004013.jpg: corrupt JPEG restored and saved
val: /kaggle/working/lodo_ready/ruod/images/train_004748.jpg: corrupt JPEG restored and saved
val: /kaggle/working/lodo_ready/ruod/images/train_006347.jpg: corrupt JPEG restored and saved
val: /kaggle/working/lodo_ready/ruod/images/train_007262.jpg: corrupt JPEG restored and saved
val: /kaggle/working/lodo_ready/ruod/images/train_008836.jpg: corrupt JPEG restored and saved


,Training domain,Unseen test domain,Precision (%),Recall (%),mAP50 (%),mAP50-95 (%)
0,Brackish,ruod,2.859,4.568,0.525,0.205


/kaggle/working/single_brackish_external_tests/single_brackish_to_ruod_metrics.csv

/kaggle/working/single_brackish_external_tests/single_brackish_to_ruod_details.json


BOTH EXTERNAL TESTS COMPLETE


,Training domain,Unseen test domain,Precision (%),Recall (%),mAP50 (%),mAP50-95 (%)
0,Brackish,norfisk,6.906,6.249,1.482,0.356
1,Brackish,ruod,2.859,4.568,0.525,0.205


/kaggle/working/single_brackish_external_tests/single_brackish_external_metrics.csv

## RUOD-only training

In [ ]:
from pathlib import Path
import json
import shutil
import hashlib
import yaml
import pandas as pd
import torch
from ultralytics import YOLO
from IPython.display import display, FileLink

BASE = Path("/kaggle/working/lodo_ready")
PARTITIONS = BASE / "audit/fixed_partitions.csv"

RUN_NAME = "single_ruod_seed42"
PROJECT = Path("/kaggle/working/single_source_runs")
RUN_DIR = PROJECT / RUN_NAME

assert torch.cuda.is_available(), "GPU enable করো।"
assert PARTITIONS.is_file(), f"Missing: {PARTITIONS}"
assert not RUN_DIR.exists(), (
    f"Run already exists: {RUN_DIR}\n"
    "আগের run আছে কি না দেখো; একই training আবার চালিও না।"
)

# Reuse existing partitions; do not create a new random split.
manifest = pd.read_csv(PARTITIONS)
ruod = manifest[
    manifest["domain"].str.lower().eq("ruod")
].copy()

assert len(ruod) == 13999, f"Unexpected RUOD count: {len(ruod)}"
assert not ruod.duplicated(["domain", "key"]).any()

train = ruod[ruod["source_partition"].eq("train")].copy()
val = ruod[ruod["source_partition"].eq("val")].copy()

assert len(train) == 11219, f"Unexpected train count: {len(train)}"
assert len(val) == 2780, f"Unexpected val count: {len(val)}"
assert set(train["key"]).isdisjoint(set(val["key"]))

if "sha256" in ruod.columns:
    assert set(train["sha256"].dropna()).isdisjoint(
        set(val["sha256"].dropna())
    ), "Duplicate hashes across train/validation."

for column in ["image_path", "label_path"]:
    missing = [
        str(p) for p in ruod[column]
        if not Path(str(p)).is_file()
    ]
    assert not missing, f"Missing {column}: {missing[:5]}"

CONFIG_DIR = BASE / "single_source_configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

train_list = CONFIG_DIR / f"{RUN_NAME}_train.txt"
val_list = CONFIG_DIR / f"{RUN_NAME}_val.txt"

train_list.write_text(
    "\n".join(train["image_path"].astype(str)) + "\n"
)
val_list.write_text(
    "\n".join(val["image_path"].astype(str)) + "\n"
)

config_path = CONFIG_DIR / f"{RUN_NAME}.yaml"
config_path.write_text(yaml.safe_dump({
    "path": str(BASE),
    "train": str(train_list),
    "val": str(val_list),
    "names": {0: "fish"},
}, sort_keys=False))

print(f"RUOD train: {len(train)}")
print(f"RUOD source validation: {len(val)}")
print("Brackish and NorFisk are excluded from training/validation.")

# Fresh COCO-pretrained initialization, same baseline settings.
model = YOLO("yolo11n.pt")

model.train(
    data=str(config_path),
    project=str(PROJECT),
    name=RUN_NAME,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    seed=42,
    deterministic=True,
    patience=30,
    save=True,
    save_period=5,
    plots=True,
    exist_ok=False,
)

# Save the actual output directory and experiment records.
RUN_DIR = Path(model.trainer.save_dir)

ruod.to_csv(RUN_DIR / "source_partitions.csv", index=False)
shutil.copy2(config_path, RUN_DIR / "dataset_config.yaml")
shutil.copy2(train_list, RUN_DIR / "source_train.txt")
shutil.copy2(val_list, RUN_DIR / "source_val.txt")

best = RUN_DIR / "weights/best.pt"
assert best.is_file(), "Best checkpoint missing."

identity = {
    "run_name": RUN_NAME,
    "training_domain": "ruod",
    "seed": 42,
    "train_images": len(train),
    "source_validation_images": len(val),
    "validation_role": "Source validation used for model selection",
    "external_test_performed": False,
    "best_checkpoint_sha256": hashlib.sha256(
        best.read_bytes()
    ).hexdigest(),
}
(RUN_DIR / "experiment_identity.json").write_text(
    json.dumps(identity, indent=2)
)

# Individual downloads, so ZIP download is not required.
files = {
    "single_ruod_best.pt": best,
    "single_ruod_last.pt": RUN_DIR / "weights/last.pt",
    "single_ruod_training_results.csv": RUN_DIR / "results.csv",
    "single_ruod_training_settings.yaml": RUN_DIR / "args.yaml",
    "single_ruod_training_curves.png": RUN_DIR / "results.png",
    "single_ruod_source_partitions.csv": RUN_DIR / "source_partitions.csv",
    "single_ruod_experiment_identity.json": RUN_DIR / "experiment_identity.json",
}

print("\nTRAINING COMPLETE — download these files:")
for filename, source in files.items():
    if source.is_file():
        destination = Path("/kaggle/working") / filename
        shutil.copy2(source, destination)
        display(FileLink(str(destination)))

print("\nBest checkpoint:", best)
print("Next: unseen Brackish and NorFisk evaluation.")